# Deep Learning Architectures for Pneumonia Detection: Comprehensive XAI Analysis

**XAI Analysis Notebook**

**Description:** This notebook provides comprehensive explainable AI (XAI) analysis for trained pneumonia detection models using GradCAM, LIME, and SHAP. It compares XAI methods across different architectures and resolutions, analyzes misclassifications, and provides similarity metrics between explanation methods.

**Main Features:**
- Comparison across architectures (AlexNet, ResNet18, ResNet50, DenseNet169)
- Resolution comparison (28px, 64px, 128px)  
- XAI method comparison (GradCAM, LIME, SHAP)
- Misclassification analysis
- Memory-efficient single-model loading
- Comprehensive similarity metrics

Authors:
- Rafaela Abrunhosa, 107658
- Miguel Pinto, 107449

# External Libraries

In [1]:
# Import utility libraries
import os
import sys
import numpy as np
import random
import gc
sys.path.append('..')  # Add parent directory to path

# Import deep learning libraries
import tensorflow as tf
from tensorflow import keras

# Suppress warnings
import warnings
warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

# Import custom utilities
# from utils.xai.xai_data_utils import *
# from utils.xai.xai_methods_utils import *
# from utils.xai.xai_plots_utils import *
from utils.xai_utils import *

print(f"TensorFlow version: {tf.__version__}")
print("XAI utilities loaded successfully!")

2025-06-21 23:07:34.903084: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-06-21 23:07:34.915814: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1750543654.930704  650255 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1750543654.934807  650255 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-06-21 23:07:34.949948: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instr

TensorFlow version: 2.18.0
XAI utilities loaded successfully!


# Configuration

In [2]:
# Configuration
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)

# Paths
DATA_PATH = '../data/'
MODEL_PATH = '../output/models/'
OUTPUT_PATH = '../output/xai_results/'

# XAI Configuration
NUM_SAMPLES_TO_ANALYZE = 10  # Number of images to analyze per class
NUM_LIME_SAMPLES = 3  # Limit LIME to fewer samples (it's slow)

# Available models from models.txt
AVAILABLE_MODELS = [
    'alexnet_augmented_b96_lr0.001_dr0.5_pneumonia_model.keras',
    'alexnet_tuned_augmented_b96_lr0.001_dr0.5_64px_pneumonia_model.keras',
    #'alexnet_tuned_augmented_b96_lr0.001_dr0.5_128px_pneumonia_model.keras',

    #'densenet_169_b96_lr0.001_dr0.5_pneumonia_model.keras',
    #'densenet_169_tuned_b96_lr0.001_dr0.5_64px_pneumonia_model.keras',
    #'densenet_169_tuned_b96_lr0.001_dr0.5_128px_pneumonia_model.keras',

    'resnet_18_augmented_b64_lr0.001_dr0.5_pneumonia_model.keras',
    'resnet_18_tuned_augmented_b64_lr0.001_dr0.5_64px_pneumonia_model.keras',
    #'resnet_18_tuned_augmented_b64_lr0.001_dr0.5_128px_pneumonia_model.keras',

    #'resnet_50_augmented_b32_lr0.001_dr0.3_pneumonia_model.keras',
    #'resnet_50_tuned_augmented_b32_lr0.001_dr0.3_64px_pneumonia_model.keras'
    #'resnet_50_tuned_augmented_b32_lr0.001_dr0.3_128px_pneumonia_model.keras',
]

# Create output directory
os.makedirs(OUTPUT_PATH, exist_ok=True)
os.makedirs(f"{OUTPUT_PATH}/predictions", exist_ok=True)
os.makedirs(f"{OUTPUT_PATH}/comparisons", exist_ok=True)

print(f"Configuration loaded!")
print(f"Will analyze {len(AVAILABLE_MODELS)} models")
print(f"Output directory: {OUTPUT_PATH}")

# Check which models exist
existing_models = []
for model_file in AVAILABLE_MODELS:
    model_path = os.path.join(MODEL_PATH, model_file)
    if os.path.exists(model_path):
        existing_models.append(model_file)
        print(f"✓ {model_file}")
    else:
        print(f"✗ {model_file} (missing)")

print(f"\nFound {len(existing_models)} existing models out of {len(AVAILABLE_MODELS)}")

Configuration loaded!
Will analyze 4 models
Output directory: ../output/xai_results/
✓ alexnet_augmented_b96_lr0.001_dr0.5_pneumonia_model.keras
✓ alexnet_tuned_augmented_b96_lr0.001_dr0.5_64px_pneumonia_model.keras
✓ resnet_18_augmented_b64_lr0.001_dr0.5_pneumonia_model.keras
✓ resnet_18_tuned_augmented_b64_lr0.001_dr0.5_64px_pneumonia_model.keras

Found 4 existing models out of 4


# Step 1: Generate Predictions for All Models

In [3]:
# Step 1: Generating Predictions for All Models

print("="*80)
print("STEP 1: GENERATING PREDICTIONS FOR ALL MODELS")
print("="*80)

# This step processes one model at a time to save memory
predictions_files = {}

for model_file in existing_models:
    print(f"\nProcessing: {model_file}")
    
    # Parse model information
    model_info = parse_model_info(model_file)
    print(f"Architecture: {model_info['architecture']}")
    print(f"Resolution: {model_info['resolution']}px")
    print(f"Is tuned: {model_info['is_tuned']}")
    
    # Check if predictions already exist
    pred_file = f"{OUTPUT_PATH}/predictions/{model_info['architecture']}_{model_info['resolution']}px_predictions.pkl"
    if os.path.exists(pred_file):
        print(f"✓ Predictions already exist: {pred_file}")
        predictions_files[(model_info['architecture'], model_info['resolution'])] = pred_file
        continue
    
    # Load test data for this resolution
    test_images, test_labels = load_test_data(DATA_PATH, model_info['resolution'])
    print(f"Test data loaded: {test_images.shape}")
    
    # Load and predict with model
    model_path = os.path.join(MODEL_PATH, model_file)
    model = tf.keras.models.load_model(model_path)
    print(f"Model loaded successfully!")
    
    # Save predictions
    pred_file = save_model_predictions(model, test_images, test_labels, model_info, f"{OUTPUT_PATH}/predictions")
    predictions_files[(model_info['architecture'], model_info['resolution'])] = pred_file
    
    # Clear memory
    del model
    del test_images, test_labels
    gc.collect()
    tf.keras.backend.clear_session()
    
    print(f"✓ Predictions saved and memory cleared")

print(f"\n✓ All predictions generated!")
print(f"Prediction files: {len(predictions_files)}")

# Step 2: Load Sample Images for XAI Analysis

print("="*80)
print("STEP 2: LOADING SAMPLE IMAGES FOR XAI ANALYSIS")
print("="*80)

# Load sample images for each resolution
sample_images = {}
sample_labels = {}
sample_indices = {}

resolutions = list(set([res for (arch, res) in predictions_files.keys()]))
print(f"Resolutions to analyze: {resolutions}")

for resolution in resolutions:
    print(f"\nLoading {resolution}px samples...")
    
    # Load test data
    test_images, test_labels = load_test_data(DATA_PATH, resolution)
    
    # Select samples
    selected_images, selected_labels, selected_indices = select_sample_images(
        test_images, test_labels, NUM_SAMPLES_TO_ANALYZE, RANDOM_SEED
    )
    
    sample_images[resolution] = selected_images
    sample_labels[resolution] = selected_labels
    sample_indices[resolution] = selected_indices
    
    print(f"✓ Selected {len(selected_images)} samples for {resolution}px")
    
    # Clear full test data to save memory
    del test_images, test_labels

print(f"\n✓ Sample images loaded for all resolutions!")

# Display sample selection summary
class_names = ['Normal', 'Pneumonia']
for resolution in resolutions:
    print(f"\n{resolution}px samples:")
    for i, class_name in enumerate(class_names):
        count = np.sum(sample_labels[resolution] == i)
        print(f"  {class_name}: {count} images")

STEP 1: GENERATING PREDICTIONS FOR ALL MODELS

Processing: alexnet_augmented_b96_lr0.001_dr0.5_pneumonia_model.keras
Architecture: AlexNet
Resolution: 28px
Is tuned: False
Test data loaded: (624, 28, 28, 1)


I0000 00:00:1750543659.154687  650255 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 2787 MB memory:  -> device: 0, name: NVIDIA GeForce GTX 1650 Ti with Max-Q Design, pci bus id: 0000:01:00.0, compute capability: 7.5


Model loaded successfully!


I0000 00:00:1750543661.121204  650552 service.cc:148] XLA service 0x7c33b001dbb0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1750543661.121251  650552 service.cc:156]   StreamExecutor device (0): NVIDIA GeForce GTX 1650 Ti with Max-Q Design, Compute Capability 7.5
2025-06-21 23:07:41.129953: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:268] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1750543661.204323  650552 cuda_dnn.cc:529] Loaded cuDNN version 90300
2025-06-21 23:07:41.455868: I external/local_xla/xla/service/gpu/autotuning/conv_algorithm_picker.cc:557] Omitted potentially buggy algorithm eng14{k25=0} for conv (f32[32,96,28,28]{3,2,1,0}, u8[0]{0}) custom-call(f32[32,1,28,28]{3,2,1,0}, f32[96,1,3,3]{3,2,1,0}, f32[96]{0}), window={size=3x3 pad=1_1x1_1}, dim_labels=bf01_oi01->bf01, custom_call_target="__cudnn$convBiasActivationForward", backend_config={"cud

✓ Predictions saved: AlexNet_28px_predictions.pkl (Accuracy: 0.856)
✓ Predictions saved and memory cleared

Processing: alexnet_tuned_augmented_b96_lr0.001_dr0.5_64px_pneumonia_model.keras
Architecture: AlexNet
Resolution: 64px
Is tuned: True
Test data loaded: (624, 64, 64, 1)
Model loaded successfully!


...

✓ Predictions saved: AlexNet_64px_predictions.pkl (Accuracy: 0.859)
✓ Predictions saved and memory cleared

Processing: resnet_18_augmented_b64_lr0.001_dr0.5_pneumonia_model.keras
Architecture: ResNet18
Resolution: 28px
Is tuned: False
Test data loaded: (624, 28, 28, 1)
Model loaded successfully!


2025-06-21 23:07:54.696074: I external/local_xla/xla/service/gpu/autotuning/conv_algorithm_picker.cc:557] Omitted potentially buggy algorithm eng14{k25=0} for conv (f32[32,64,14,14]{3,2,1,0}, u8[0]{0}) custom-call(f32[32,64,14,14]{3,2,1,0}, f32[64,64,3,3]{3,2,1,0}, f32[64]{0}), window={size=3x3 pad=1_1x1_1}, dim_labels=bf01_oi01->bf01, custom_call_target="__cudnn$convBiasActivationForward", backend_config={"cudnn_conv_backend_config":{"activation_mode":"kNone","conv_result_scale":1,"leakyrelu_alpha":0,"side_input_scale":0},"force_earliest_schedule":false,"operation_queue_id":"0","wait_on_operation_queues":[]}
2025-06-21 23:07:54.884534: I external/local_xla/xla/service/gpu/autotuning/conv_algorithm_picker.cc:557] Omitted potentially buggy algorithm eng14{k25=0} for conv (f32[32,128,7,7]{3,2,1,0}, u8[0]{0}) custom-call(f32[32,128,7,7]{3,2,1,0}, f32[128,128,3,3]{3,2,1,0}, f32[128]{0}), window={size=3x3 pad=1_1x1_1}, dim_labels=bf01_oi01->bf01, custom_call_target="__cudnn$convBiasActivati

✓ Predictions saved: ResNet18_28px_predictions.pkl (Accuracy: 0.880)
✓ Predictions saved and memory cleared

Processing: resnet_18_tuned_augmented_b64_lr0.001_dr0.5_64px_pneumonia_model.keras
Architecture: ResNet18
Resolution: 64px
Is tuned: True
Test data loaded: (624, 64, 64, 1)
Model loaded successfully!


2025-06-21 23:08:00.911424: I external/local_xla/xla/service/gpu/autotuning/conv_algorithm_picker.cc:557] Omitted potentially buggy algorithm eng14{k25=0} for conv (f32[32,64,64,64]{3,2,1,0}, u8[0]{0}) custom-call(f32[32,1,64,64]{3,2,1,0}, f32[64,1,3,3]{3,2,1,0}, f32[64]{0}), window={size=3x3 pad=1_1x1_1}, dim_labels=bf01_oi01->bf01, custom_call_target="__cudnn$convBiasActivationForward", backend_config={"cudnn_conv_backend_config":{"activation_mode":"kNone","conv_result_scale":1,"leakyrelu_alpha":0,"side_input_scale":0},"force_earliest_schedule":false,"operation_queue_id":"0","wait_on_operation_queues":[]}
2025-06-21 23:08:00.937443: I external/local_xla/xla/service/gpu/autotuning/conv_algorithm_picker.cc:557] Omitted potentially buggy algorithm eng14{k25=0} for conv (f32[32,64,32,32]{3,2,1,0}, u8[0]{0}) custom-call(f32[32,64,32,32]{3,2,1,0}, f32[64,64,3,3]{3,2,1,0}, f32[64]{0}), window={size=3x3 pad=1_1x1_1}, dim_labels=bf01_oi01->bf01, custom_call_target="__cudnn$convBiasActivationF

✓ Predictions saved: ResNet18_64px_predictions.pkl (Accuracy: 0.909)
✓ Predictions saved and memory cleared

✓ All predictions generated!
Prediction files: 4
STEP 2: LOADING SAMPLE IMAGES FOR XAI ANALYSIS
Resolutions to analyze: [64, 28]

Loading 64px samples...
✓ Selected 20 samples for 64px

Loading 28px samples...
✓ Selected 20 samples for 28px

✓ Sample images loaded for all resolutions!

64px samples:
  Normal: 10 images
  Pneumonia: 10 images

28px samples:
  Normal: 10 images
  Pneumonia: 10 images


# Step 3: Generate XAI Analysis for Each Model

In [4]:
# Step 3: Generate XAI Analysis for Each Model

print("="*80)
print("STEP 3: GENERATING XAI ANALYSIS FOR EACH MODEL")
print("="*80)

# This will store all XAI results
xai_results = {}

for model_file in existing_models:
    print(f"\n{'='*50}")
    print(f"Analyzing: {model_file}")
    print(f"{'='*50}")
    
    # Parse model information
    model_info = parse_model_info(model_file)
    resolution = model_info['resolution']
    architecture = model_info['architecture']
    
    # Load model
    model_path = os.path.join(MODEL_PATH, model_file)
    model = tf.keras.models.load_model(model_path)
    print(f"✓ Model loaded: {architecture} ({resolution}px)")
    
    # Get sample images for this resolution
    images = sample_images[resolution]
    labels = sample_labels[resolution]
    
    # Get model predictions for samples
    predictions = model.predict(images, verbose=0)
    pred_classes = np.argmax(predictions, axis=1)
    pred_probs = np.max(predictions, axis=1)
    
    print(f"Sample predictions accuracy: {np.sum(pred_classes == labels)}/{len(labels)} ({np.sum(pred_classes == labels)/len(labels)*100:.1f}%)")
    
    # Initialize results structure
    model_results = {
        'pred_classes': pred_classes,
        'true_classes': labels,
        'pred_probs': pred_probs,
        'gradcam': None,
        'lime': None,
        'shap': None
    }
    
    # 1. GradCAM Analysis
    print("\n1. Applying GradCAM...")
    try:
        gradcam_results, used_layer = apply_gradcam_analysis(model, images, pred_classes)
        model_results['gradcam'] = gradcam_results
        print(f"✓ GradCAM completed using layer: {used_layer}")
    except Exception as e:
        print(f"✗ GradCAM failed: {e}")
        model_results['gradcam'] = [None] * len(images)
    
    # 2. LIME Analysis (limited samples)
    print(f"\n2. Applying LIME (first {NUM_LIME_SAMPLES} samples)...")
    try:
        lime_results = apply_lime_analysis(model, images[:NUM_LIME_SAMPLES])
        # Extend to full length with None values
        full_lime_results = lime_results + [None] * (len(images) - len(lime_results))
        model_results['lime'] = full_lime_results
        print(f"✓ LIME completed for {len(lime_results)} samples")
    except Exception as e:
        print(f"✗ LIME failed: {e}")
        model_results['lime'] = [None] * len(images)
    
    # 3. SHAP Analysis
    print("\n3. Applying SHAP...")
    try:
        shap_values = apply_shap_analysis(model, images)
        model_results['shap'] = shap_values
        if shap_values is not None:
            print("✓ SHAP completed")
        else:
            print("✗ SHAP failed")
    except Exception as e:
        print(f"✗ SHAP failed: {e}")
        model_results['shap'] = None
    
    # Store results using a tuple key (architecture, resolution)
    xai_results[(architecture, resolution)] = model_results
    
    # Clear memory
    del model
    gc.collect()
    tf.keras.backend.clear_session()
    
    print(f"✓ XAI analysis completed and memory cleared")

print(f"\n✓ All XAI analyses completed!")
print(f"Analyzed models: {len(xai_results)}")

STEP 3: GENERATING XAI ANALYSIS FOR EACH MODEL

Analyzing: alexnet_augmented_b96_lr0.001_dr0.5_pneumonia_model.keras
✓ Model loaded: AlexNet (28px)


2025-06-21 23:08:07.029767: I external/local_xla/xla/service/gpu/autotuning/conv_algorithm_picker.cc:557] Omitted potentially buggy algorithm eng14{k25=0} for conv (f32[20,96,28,28]{3,2,1,0}, u8[0]{0}) custom-call(f32[20,1,28,28]{3,2,1,0}, f32[96,1,3,3]{3,2,1,0}, f32[96]{0}), window={size=3x3 pad=1_1x1_1}, dim_labels=bf01_oi01->bf01, custom_call_target="__cudnn$convBiasActivationForward", backend_config={"cudnn_conv_backend_config":{"activation_mode":"kNone","conv_result_scale":1,"leakyrelu_alpha":0,"side_input_scale":0},"force_earliest_schedule":false,"operation_queue_id":"0","wait_on_operation_queues":[]}
2025-06-21 23:08:07.039452: I external/local_xla/xla/service/gpu/autotuning/conv_algorithm_picker.cc:557] Omitted potentially buggy algorithm eng14{k25=0} for conv (f32[20,256,14,14]{3,2,1,0}, u8[0]{0}) custom-call(f32[20,96,14,14]{3,2,1,0}, f32[256,96,3,3]{3,2,1,0}, f32[256]{0}), window={size=3x3 pad=1_1x1_1}, dim_labels=bf01_oi01->bf01, custom_call_target="__cudnn$convBiasActivati

Sample predictions accuracy: 14/20 (70.0%)

1. Applying GradCAM...
Using layer 'conv2d_14' for GradCAM analysis
✓ GradCAM completed using layer: conv2d_14

2. Applying LIME (first 3 samples)...
Processing LIME for image 1/3...


  0%|          | 0/1000 [00:00<?, ?it/s]2025-06-21 23:08:10.604192: I external/local_xla/xla/service/gpu/autotuning/conv_algorithm_picker.cc:557] Omitted potentially buggy algorithm eng14{k25=0} for conv (f32[10,256,14,14]{3,2,1,0}, u8[0]{0}) custom-call(f32[10,96,14,14]{3,2,1,0}, f32[256,96,3,3]{3,2,1,0}, f32[256]{0}), window={size=3x3 pad=1_1x1_1}, dim_labels=bf01_oi01->bf01, custom_call_target="__cudnn$convBiasActivationForward", backend_config={"cudnn_conv_backend_config":{"activation_mode":"kNone","conv_result_scale":1,"leakyrelu_alpha":0,"side_input_scale":0},"force_earliest_schedule":false,"operation_queue_id":"0","wait_on_operation_queues":[]}
2025-06-21 23:08:10.699585: I external/local_xla/xla/service/gpu/autotuning/conv_algorithm_picker.cc:557] Omitted potentially buggy algorithm eng14{k25=0} for conv (f32[10,384,7,7]{3,2,1,0}, u8[0]{0}) custom-call(f32[10,256,7,7]{3,2,1,0}, f32[384,256,3,3]{3,2,1,0}, f32[384]{0}), window={size=3x3 pad=1_1x1_1}, dim_labels=bf01_oi01->bf01, c

Processing LIME for image 2/3...


...

LIME failed for image 1: Graph execution error:

Detected at node convolution defined at (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main

  File "<frozen runpy>", line 88, in _run_code

  File "/home/miguel/Desktop/CAA/CAA_Project2/venv/lib/python3.12/site-packages/ipykernel_launcher.py", line 18, in <module>

  File "/home/miguel/Desktop/CAA/CAA_Project2/venv/lib/python3.12/site-packages/traitlets/config/application.py", line 1075, in launch_instance

  File "/home/miguel/Desktop/CAA/CAA_Project2/venv/lib/python3.12/site-packages/ipykernel/kernelapp.py", line 739, in start

  File "/home/miguel/Desktop/CAA/CAA_Project2/venv/lib/python3.12/site-packages/tornado/platform/asyncio.py", line 205, in start

  File "/usr/lib/python3.12/asyncio/base_events.py", line 641, in run_forever

  File "/usr/lib/python3.12/asyncio/base_events.py", line 1987, in _run_once

  File "/usr/lib/python3.12/asyncio/events.py", line 88, in _run

  File "/home/miguel/Desktop/C

  0%|          | 0/1000 [00:00<?, ?it/s]2025-06-21 23:09:36.625589: W tensorflow/core/framework/op_kernel.cc:1841] OP_REQUIRES failed at conv_ops.cc:61 : INVALID_ARGUMENT: Depth of output must be a multiple of the number of groups: 64 vs 3

Stack trace for op definition: 
File "<frozen runpy>", line 198, in _run_module_as_main
File "<frozen runpy>", line 88, in _run_code
File "/home/miguel/Desktop/CAA/CAA_Project2/venv/lib/python3.12/site-packages/ipykernel_launcher.py", line 18, in <module>
File "/home/miguel/Desktop/CAA/CAA_Project2/venv/lib/python3.12/site-packages/traitlets/config/application.py", line 1075, in launch_instance
File "/home/miguel/Desktop/CAA/CAA_Project2/venv/lib/python3.12/site-packages/ipykernel/kernelapp.py", line 739, in start
File "/home/miguel/Desktop/CAA/CAA_Project2/venv/lib/python3.12/site-packages/tornado/platform/asyncio.py", line 205, in start
File "/usr/lib/python3.12/asyncio/base_events.py", line 641, in run_forever
File "/usr/lib/python3.12/asyncio/ba

LIME failed for image 2: Graph execution error:

Detected at node convolution defined at (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main

  File "<frozen runpy>", line 88, in _run_code

  File "/home/miguel/Desktop/CAA/CAA_Project2/venv/lib/python3.12/site-packages/ipykernel_launcher.py", line 18, in <module>

  File "/home/miguel/Desktop/CAA/CAA_Project2/venv/lib/python3.12/site-packages/traitlets/config/application.py", line 1075, in launch_instance

  File "/home/miguel/Desktop/CAA/CAA_Project2/venv/lib/python3.12/site-packages/ipykernel/kernelapp.py", line 739, in start

  File "/home/miguel/Desktop/CAA/CAA_Project2/venv/lib/python3.12/site-packages/tornado/platform/asyncio.py", line 205, in start

  File "/usr/lib/python3.12/asyncio/base_events.py", line 641, in run_forever

  File "/usr/lib/python3.12/asyncio/base_events.py", line 1987, in _run_once

  File "/usr/lib/python3.12/asyncio/events.py", line 88, in _run

  File "/home/miguel/Desktop/C

2025-06-21 23:09:38.892139: W external/local_xla/xla/tsl/framework/bfc_allocator.cc:306] Allocator (GPU_0_bfc) ran out of memory trying to allocate 2.40GiB with freed_by_count=0. The caller indicates that this is not a failure, but this may mean that there could be performance gains if more memory were available.


✓ SHAP completed
✓ XAI analysis completed and memory cleared

✓ All XAI analyses completed!
Analyzed models: 4


# Step 4: Comprehensive XAI Comparisons

In [5]:
print("="*80)
print("STEP 4: COMPREHENSIVE XAI COMPARISONS")
print("="*80)

# 4.1: Compare by Architecture (different resolutions for same architecture)
print("\n4.1: Comparing by Architecture (Resolution Effects)")
print("-" * 50)

plot_xai_comparison_by_architecture(xai_results, sample_images, sample_labels, f"{OUTPUT_PATH}/comparisons")

# 4.2: Compare by Resolution (different architectures for same resolution)
print("\n4.2: Comparing by Resolution (Architecture Effects)")
print("-" * 50)

plot_xai_comparison_by_resolution(xai_results, sample_images, sample_labels, f"{OUTPUT_PATH}/comparisons")

print("\n✓ Comparison visualizations completed!")

STEP 4: COMPREHENSIVE XAI COMPARISONS

4.1: Comparing by Architecture (Resolution Effects)
--------------------------------------------------
Creating comparison plots for AlexNet across [28, 64]px
✓ Saved comparison plot: ../output/xai_results//comparisons/AlexNet_resolution_comparison.png
Creating comparison plots for ResNet18 across [28, 64]px
✓ Saved comparison plot: ../output/xai_results//comparisons/ResNet18_resolution_comparison.png

4.2: Comparing by Resolution (Architecture Effects)
--------------------------------------------------
Creating comparison plots for 28px across ['AlexNet', 'ResNet18']
✓ Saved comparison plot: ../output/xai_results//comparisons/28px_architecture_comparison.png
Creating comparison plots for 64px across ['AlexNet', 'ResNet18']
✓ Saved comparison plot: ../output/xai_results//comparisons/64px_architecture_comparison.png

✓ Comparison visualizations completed!


# XAI Method 2: LIME Analysis

# Step 5: Misclassification Analysis

In [6]:
print("="*80)
print("STEP 5: MISCLASSIFICATION ANALYSIS")
print("="*80)

# Load all prediction data
print("Loading prediction data...")
predictions_data = {}

for (arch, res), pred_file in predictions_files.items():
    pred_data = load_model_predictions(pred_file)
    # Use tuple key for consistency with xai_results
    predictions_data[(arch, res)] = pred_data

print(f"Loaded predictions for {len(predictions_data)} models")

# Analyze misclassifications
print("\nAnalyzing misclassifications...")
misclassification_analysis = analyze_misclassifications(
    predictions_data, sample_images, sample_labels, f"{OUTPUT_PATH}/comparisons"
)

print(f"Images misclassified by all models: {len(misclassification_analysis['misclassified_by_all'])}")
print(f"Images correctly classified by all models: {len(misclassification_analysis['correctly_classified_by_all'])}")

print("\n✓ Misclassification analysis completed!")

STEP 5: MISCLASSIFICATION ANALYSIS
Loading prediction data...
Loaded predictions for 4 models

Analyzing misclassifications...
Analyzing misclassifications across models...
Analyzing 20 samples at 64px resolution
✓ Misclassification visualization saved
Analysis complete: 10 correctly classified by all, 7 misclassified by all
Images misclassified by all models: 7
Images correctly classified by all models: 10

✓ Misclassification analysis completed!


# XAI Method 3: SHAP Analysis

# Step 6: XAI Similarity Metrics

In [7]:
print("="*80)
print("STEP 6: XAI SIMILARITY METRICS")
print("="*80)

# Calculate similarity metrics between XAI methods
print("Calculating XAI similarity metrics...")
similarity_metrics = calculate_xai_similarity_metrics(xai_results)

print("\nXAI Method Similarity Results:")
print("-" * 40)

for model_key, metrics in similarity_metrics.items():
    print(f"\n{model_key}:")
    for method_pair, values in metrics.items():
        print(f"  {method_pair}:")
        for metric_name, value in values.items():
            if not np.isnan(value):
                print(f"    {metric_name}: {value:.3f}")
            else:
                print(f"    {metric_name}: N/A")

print("\n✓ Similarity metrics calculated!")

STEP 6: XAI SIMILARITY METRICS
Calculating XAI similarity metrics...

XAI Method Similarity Results:
----------------------------------------

AlexNet_28px:
  GradCAM_vs_LIME:
    mean_cosine: N/A
    mean_pearson: N/A
    mean_spearman: N/A
    num_comparisons: 0.000
  GradCAM_vs_SHAP:
    mean_cosine: N/A
    mean_pearson: N/A
    mean_spearman: N/A
    num_comparisons: 0.000

AlexNet_64px:
  GradCAM_vs_LIME:
    mean_cosine: N/A
    mean_pearson: N/A
    mean_spearman: N/A
    num_comparisons: 0.000
  GradCAM_vs_SHAP:
    mean_cosine: N/A
    mean_pearson: N/A
    mean_spearman: N/A
    num_comparisons: 0.000

ResNet18_28px:
  GradCAM_vs_LIME:
    mean_cosine: N/A
    mean_pearson: N/A
    mean_spearman: N/A
    num_comparisons: 0.000
  GradCAM_vs_SHAP:
    mean_cosine: N/A
    mean_pearson: N/A
    mean_spearman: N/A
    num_comparisons: 0.000

ResNet18_64px:
  GradCAM_vs_LIME:
    mean_cosine: N/A
    mean_pearson: N/A
    mean_spearman: N/A
    num_comparisons: 0.000
  GradCAM_vs

In [8]:
print("="*80)
print("STEP 7: GENERATING COMPREHENSIVE ANALYSIS REPORT")
print("="*80)

# Generate and save comprehensive report
print("Generating comprehensive analysis report...")
analysis_report = save_xai_analysis_report(
    xai_results, similarity_metrics, misclassification_analysis, OUTPUT_PATH
)

print("\nANALYSIS SUMMARY:")
print("="*50)

# Model Performance Summary
print("\nModel Performance Summary:")
print("-" * 30)
for model_key, summary in analysis_report['model_summaries'].items():
    print(f"\n{model_key}:")
    print(f"  Accuracy: {summary['model_accuracy']:.3f}")
    print(f"  GradCAM Success: {summary['gradcam_success_rate']:.3f}")
    print(f"  LIME Success: {summary['lime_success_rate']:.3f}")
    print(f"  SHAP Success: {summary['shap_success_rate']:.3f}")

# XAI Method Comparison Summary
print(f"\nXAI Method Effectiveness:")
print("-" * 30)
all_gradcam_success = [s['gradcam_success_rate'] for s in analysis_report['model_summaries'].values()]
all_lime_success = [s['lime_success_rate'] for s in analysis_report['model_summaries'].values()]
all_shap_success = [s['shap_success_rate'] for s in analysis_report['model_summaries'].values()]

print(f"GradCAM: {np.mean(all_gradcam_success):.3f} ± {np.std(all_gradcam_success):.3f} success rate")
print(f"LIME: {np.mean(all_lime_success):.3f} ± {np.std(all_lime_success):.3f} success rate")
print(f"SHAP: {np.mean(all_shap_success):.3f} ± {np.std(all_shap_success):.3f} success rate")

# Similarity Analysis Summary
print(f"\nXAI Method Similarity (where available):")
print("-" * 40)
all_cosine_similarities = []
all_pearson_correlations = []

for model_metrics in similarity_metrics.values():
    for method_pair, values in model_metrics.items():
        if not np.isnan(values['mean_cosine']):
            all_cosine_similarities.append(values['mean_cosine'])
        if not np.isnan(values['mean_pearson']):
            all_pearson_correlations.append(values['mean_pearson'])

if all_cosine_similarities:
    print(f"Average Cosine Similarity: {np.mean(all_cosine_similarities):.3f} ± {np.std(all_cosine_similarities):.3f}")
if all_pearson_correlations:
    print(f"Average Pearson Correlation: {np.mean(all_pearson_correlations):.3f} ± {np.std(all_pearson_correlations):.3f}")

print(f"\n✓ Comprehensive analysis completed!")
print(f"📁 All results saved to: {OUTPUT_PATH}")
print(f"📊 Comparison plots saved to: {OUTPUT_PATH}/comparisons/")
print(f"🔍 Analysis report: {OUTPUT_PATH}/xai_analysis_report.json")

STEP 7: GENERATING COMPREHENSIVE ANALYSIS REPORT
Generating comprehensive analysis report...
✓ Analysis report saved: ../output/xai_results/xai_analysis_report.json

ANALYSIS SUMMARY:

Model Performance Summary:
------------------------------

AlexNet_28px:
  Accuracy: 0.700
  GradCAM Success: 1.000
  LIME Success: 0.150
  SHAP Success: 1.000

AlexNet_64px:
  Accuracy: 0.800
  GradCAM Success: 1.000
  LIME Success: 0.150
  SHAP Success: 1.000

ResNet18_28px:
  Accuracy: 0.650
  GradCAM Success: 1.000
  LIME Success: 0.000
  SHAP Success: 1.000

ResNet18_64px:
  Accuracy: 0.750
  GradCAM Success: 1.000
  LIME Success: 0.000
  SHAP Success: 1.000

XAI Method Effectiveness:
------------------------------
GradCAM: 1.000 ± 0.000 success rate
LIME: 0.075 ± 0.075 success rate
SHAP: 1.000 ± 0.000 success rate

XAI Method Similarity (where available):
----------------------------------------

✓ Comprehensive analysis completed!
📁 All results saved to: ../output/xai_results/
📊 Comparison plots s

## Step 7: Generate Comprehensive Analysis Report

In [9]:
print("\n" + "="*80)
print("ANALYSIS COMPLETED - CHECK OUTPUT FOLDERS FOR DETAILED RESULTS")
print("="*80)
print(f"📁 All results saved to: {OUTPUT_PATH}")
print(f"📊 Comparison plots saved to: {OUTPUT_PATH}/comparisons/")
print(f"🔍 Analysis report: {OUTPUT_PATH}/xai_analysis_report.json")


ANALYSIS COMPLETED - CHECK OUTPUT FOLDERS FOR DETAILED RESULTS
📁 All results saved to: ../output/xai_results/
📊 Comparison plots saved to: ../output/xai_results//comparisons/
🔍 Analysis report: ../output/xai_results//xai_analysis_report.json
